# Raman protein perturbation pub

In [ ]:
from pathlib import Path

import arcadia_pycolor as apc
import numpy as np
import pandas as pd
import plotly.colors as pcolors
import plotly.express as px
import plotly.graph_objects as go
import ramanspy as rp
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit

from raman_analysis.utils import collect_spectra_from_dir, plot_spectra_in_dir

apc.plotly.setup()
%load_ext autoreload
%autoreload 2

raman_chem_perturbation_dir = Path('data/raman-chemical-perturbations')
raman_ph_perturbation_dir = Path('data/raman-ph-perturbations')
plot_dir_htmls = Path('plots/Raman/html')
plot_dir_pdf = Path('plots/Raman/pdf')

plot_scale = 3
fig_width = 1000
fig_height=400

## Chemical perturbations


### Colormap for chemical perturbations

* No perturbation: Chateau (b9afa7)
* TCEP: Canary (f7b846)
* EDTA: Aegean (5088c5)
* Both: Seaweed (3b9886)

In [ ]:
chem_perturbation_palette = apc.Palette("chem_perturbations", [apc.chateau, apc.canary, apc.aegean, apc.lime])
chem_perturbation_palette

In [ ]:
def generate_subtract_dict(samples):
    """ Generate a dictionary mapping sample names to their buffer names for subtraction. """
    subtract_dict = {}
    for name in samples:
        date = name.split('/')[0]
        buffer = name.split(' in ')[-1]
        subtract_dict[name] = f"{date}/{buffer}"
    return subtract_dict

In [ ]:
data_dir = raman_chem_perturbation_dir

data_list, sample_df = collect_spectra_from_dir(data_dir, 'instantraman', recursive=True)

pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        # rp.preprocessing.baseline.ASPLS(lam=5e4),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
        # rp.preprocessing.normalise.AUC(),
    ]
)


### All plottable proteins

In [ ]:
buffers = ['Tris', 'Tris+TCEP', 'Tris+EDTA', 'Tris+TCEP+EDTA']
all_samples = sample_df['Name'].str.split('/').str[-1].str.split(' in ').str[0].unique()
all_samples = [s for s in all_samples if not any(b in s for b in buffers)]
all_samples

In [ ]:
xranges = {'FAF-BSA [10 mgml]': [None, [360, 700], [1150,1378]],
           'DNAse [12.5 mgml]': [None, [360, 700], [1150,1378]],
           'CytC [10 mgml]': [None, [360, 700], [1500,1650]]
           }
yranges = {'FAF-BSA [10 mgml]': [None, [-200,800], [-500, 2500]],
           'DNAse [12.5 mgml]': [None, [-300, 1500], None],
           'CytC [10 mgml]': [None, None, None]}

fig_heights = {'FAF-BSA [10 mgml]': [fig_height, fig_height, fig_height],
               'DNAse [12.5 mgml]': [fig_height, fig_height, fig_height],
               'CytC [10 mgml]': [fig_height, fig_height, fig_height]}
fig_widths = {'FAF-BSA [10 mgml]': [fig_width, fig_width*0.5, fig_width*0.5],
              'DNAse [12.5 mgml]': [fig_width, fig_width*0.5, fig_width*0.5],
              'CytC [10 mgml]': [fig_width, fig_width*0.5, fig_width*0.5]}

n_groups = sample_df['Group ID'].nunique()

buffer_dict = {'FAF-BSA [10 mgml]': buffers[:-1], # exclude TCEP+EDTA condition for BSA since it causes precipitation
           'DNAse [12.5 mgml]': buffers,
           'CytC [10 mgml]': buffers}


for protein in xranges.keys():
    for i, (xrange, yrange) in enumerate(zip(xranges[protein], yranges[protein])):
        include_samples = [n for n in sample_df['Name'] if (any(n.endswith(buf) for buf in buffer_dict[protein]) and protein in n)]
        subtract_id = generate_subtract_dict(include_samples)
        include_samples += list(subtract_id.values())
        f, names, rp_container = plot_spectra_in_dir(data_list,
                            col_to_plot = "Intensity (dark-subtracted)",
                            data_dir = data_dir,
                            subtract_id=subtract_id, # None if ('BSA' in protein) and ('Tris+TCEP+EDTA' in buffer_subset) else subtract_id,
                            rp_pipeline=pipeline,
                            colors=chem_perturbation_palette,
                            include=include_samples,
                            height=fig_heights[protein][i],
                            width=fig_widths[protein][i],
                            xaxis_title='Wavenumber (cm⁻¹)',
                            yaxis_title='Intensity (a.u.)',
                            title=f"{protein} | {xrange}_cm-1 | {'_'.join(buffer_dict[protein])}"
                            )
        f.update_xaxes(range=xrange)
        f.update_yaxes(range=yrange)
        # f.data = tuple(trace for trace in f.data if trace.name != subtract_id)
        f.data = tuple(trace for trace in f.data if trace.name not in subtract_id.values())
        f.update_layout(showlegend=False)
        apc.plotly.style_plot(f, monospaced_axes="all")

        protein_dir_pdf = plot_dir_pdf / protein
        protein_dir_html = plot_dir_htmls / protein
        protein_dir_pdf.mkdir(parents=True, exist_ok=True)
        protein_dir_html.mkdir(parents=True, exist_ok=True)

        f.write_image(protein_dir_pdf / f"{protein}_{xrange}_cm-1.pdf", scale=plot_scale)
        f.write_html(protein_dir_html / f"{protein}_{xrange}_cm-1.html")
        f.show()



### Additional proteins

In [ ]:
additional_proteins = [s for s in all_samples if s not in xranges.keys()]
for sample_to_plot in additional_proteins:
    if sample_to_plot not in additional_proteins:
        raise ValueError(f"Sample {sample_to_plot} not found in data. Available samples: {additional_proteins}")

    include_samples = [n for n in sample_df['Name'] if n.split('/')[-1].startswith(sample_to_plot) and any(n.endswith(buf) for buf in buffers)]
    subtract_id = generate_subtract_dict(include_samples)
    include_samples += list(subtract_id.values())
    f, names, rp_container = plot_spectra_in_dir(data_list,
                        col_to_plot = "Intensity (dark-subtracted)",
                        data_dir = data_dir,
                        subtract_id=subtract_id,
                        rp_pipeline=pipeline,
                        colors=chem_perturbation_palette,
                        include=include_samples,
                        height=fig_height,
                        width=fig_width,
                        # y_offset=0.002,
                        xaxis_title='Wavenumber (cm⁻¹)',
                        yaxis_title='Intensity (a.u.)',
                        title=sample_to_plot,
                        )
    f.data = tuple(trace for trace in f.data if trace.name not in subtract_id.values())

    apc.plotly.style_plot(f, monospaced_axes="all")

    f.show()

### Buffers alone

In [ ]:
buffers

In [ ]:
include_samples = [n for n in sample_df['Name'] if (any(n == f'2025-12-10/{buf}' for buf in buffers))]

f, names, rp_container = plot_spectra_in_dir(data_list,
                    col_to_plot = "Intensity (dark-subtracted)",
                    data_dir = data_dir,
                    subtract_id=None,
                    rp_pipeline=pipeline,
                    colors=chem_perturbation_palette,
                    include=include_samples,
                    height=fig_height,
                    width=fig_width,
                    # y_offset=0.002,
                    xaxis_title='Wavenumber (cm⁻¹)',
                    yaxis_title='Intensity (a.u.)',
                    title="Buffers",
                    )


apc.plotly.style_plot(f, monospaced_axes="all")
f.write_image(plot_dir_pdf / "buffers.pdf", scale=plot_scale)
f.write_html(plot_dir_htmls / "buffers.html")
f.show()

## pH perturbations

In [ ]:
data_list, sample_df = collect_spectra_from_dir(raman_ph_perturbation_dir, 'instantraman', recursive=True)

pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        # rp.preprocessing.baseline.ASPLS(lam=5e4),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
        # rp.preprocessing.normalise.AUC(),
    ]
)

n_groups = sample_df['Group ID'].nunique()


In [ ]:
def generate_subtract_dict_pH(_replicate, sample_string):
    ph_vals = [0, 0.3, 1, 1.3, 2, 2.3, 3, 3.3, 4, 5, 6, 7.5, 8, 9, 9.5, 10, 10.7, 11, 11.7, 12, 12.7, 13, 13.7, 14]
    ph_vals_str = [str(p) for p in ph_vals]

    sample_names = sample_df['Name'].unique().tolist()
    all_protein = [n for n in sample_names if sample_string + " pH" in n and _replicate in n]

    subtraction_key = 'TCEP' if 'TCEP' in sample_string else 'buffers'
    subtract_dict = {}
    for p in all_protein:
        for ph in ph_vals_str:
            if f"{sample_string} pH {ph}" in p:
                subtract_dict[p] = [b for b in sample_names if b.endswith(f"{_replicate}/{subtraction_key} pH {ph}")][0]
    return subtract_dict


In [ ]:
ph_fig_width = 1300
ph_fig_height = 400
ph_gradient = apc.Gradient("ph_gradient", [apc.cinnabar, apc.chateau, apc.lapis])
BSA_color = apc.dragon
BSA_TCEP_color = apc.aegean

ph_to_plot = [0.3, 1, 1.3, 2, 2.3, 3, 7.5, 8, 9, 9.5, 11, 11.7, 12, 12.7, 13]
ph_traces_to_show = [2.3, 7.5, 9.5, 11.7]
ph_vals_to_show_on_colorbar = [0.3, 7.5, 13]

In [ ]:
replicate = 'rep1'
ph_to_plot_labels = [f"pH {ph_val}" for ph_val in ph_to_plot]

spectral_regions = {
    (1400, 1420): 'Carboxylate',
    (1650, 1657): 'Amide I<br>⍺-Helix',
    (1227, 1320): 'Amide III',
    (900, 940): 'C⍺-C stretch',
}

sample_names_dict = {}
sample_containers_dict = {}

for sample in ['BSA', 'BSA+TCEP']:
    subtract_dict = generate_subtract_dict_pH(replicate, sample)
    subtract_dict = {k: v for k, v in subtract_dict.items() if any(k.endswith(label) for label in ph_to_plot_labels)}

    include_samples = list(subtract_dict.keys()) + list(subtract_dict.values())
    exclude_from_plot = subtract_dict.values()

    colors = pcolors.sample_colorscale(px.colors.qualitative.Dark24, samplepoints=n_groups)

    f, names, rp_container = plot_spectra_in_dir(data_list,
                        col_to_plot = "Intensity (dark-subtracted)",
                        data_dir = raman_ph_perturbation_dir,
                        subtract_id=subtract_dict,
                        rp_pipeline=pipeline,
                        colors=colors,
                        include=include_samples,
                        height=ph_fig_height,
                        width=ph_fig_width,
                        xaxis_title='Wavenumber (cm⁻¹)',
                        yaxis_title='Intensity (a.u.)',
                        title=f"{sample} pH perturbation"
    )
    f.data = tuple(trace for trace in f.data if not any(trace.name == exclude for exclude in exclude_from_plot))
    for trace in f.data:
        if not any(str(trace.name).endswith(f"pH {ph}") for ph in ph_traces_to_show):
            trace.visible = 'legendonly'
    apc.plotly.style_plot(f, monospaced_axes="all")

    gradient = ph_gradient.resample_as_palette(len(ph_to_plot_labels))
    for i, trace in enumerate(f.data):
        trace.line.color = gradient[i]
    
    # Add a colorbar for the gradient with pH values
    colorbar_trace = go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(
            colorscale=[ [i/(len(gradient)-1), color] for i, color in enumerate(gradient) ],
            cmin=min(ph_to_plot),
            cmax=max(ph_to_plot),
            colorbar=dict(
                title='pH',
                tickvals=ph_to_plot,
                ticktext=[str(ph) for ph in ph_to_plot],
                lenmode='pixels',
                len=200,
            ),
            color=[min(ph_to_plot), max(ph_to_plot)],
            showscale=True,
            size=0.1,
        ),
        showlegend=False,
        hoverinfo='none'
    )
    colorbar_trace.marker.colorbar.update(
        x=0.01,
        y=1.05,
        xanchor='left',
        yanchor='bottom',
        orientation='h',
        thickness=5,
        lenmode='pixels',
        len=200,
        title_side='top',
        tickvals=ph_vals_to_show_on_colorbar,
        ticktext=[str(ph) for ph in ph_vals_to_show_on_colorbar]
    )
    f.add_trace(colorbar_trace)

    # Spectral region bars: x in data coords (moves with zoom), y in paper coords (stays at top)
    for (x0, x1), label in spectral_regions.items():
        f.add_shape(
            type='line',
            x0=x0, x1=x1, y0=1.04, y1=1.04,
            xref='x', yref='paper',
            line=dict(color='black', width=2),
            layer='above',
        )
        f.add_annotation(
            x=(x0 + x1) / 2, y=1.06,
            xref='x', yref='paper',
            text=label, 
            showarrow=False,
            font=dict(size=10),
            yanchor='bottom',
            xanchor='center',
        )

    f.update_layout(margin=dict(t=100))
    f.show()

    f.write_html(plot_dir_htmls / f"{sample}_ph_spectra.html")
    f.write_image(plot_dir_pdf / f"{sample}_ph_spectra.pdf", scale=plot_scale)
    
    sample_names_dict[sample] = names
    sample_containers_dict[sample] = rp_container


In [ ]:
def intensity_at(spectral_axis, spectral_data_row, wavenumber):
    idx = np.argmin(np.abs(spectral_axis - wavenumber))
    return float(spectral_data_row[idx])

def fit_gaussian_peak(spectral_axis, spectral_data_row, center, window=60):
    """Fit a Gaussian to a spectral region and return (center, fwhm). Returns (nan, nan) on failure."""
    mask = (spectral_axis >= center - window) & (spectral_axis <= center + window)
    x, y = spectral_axis[mask], spectral_data_row[mask]
    if len(x) < 4:
        return np.nan, np.nan
    gaussian = lambda x, amp, mu, sigma: amp * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))
    try:
        p0 = [y.max(), center, window / 3]
        bounds = ([0, center - window, 1], [np.inf, center + window, window])
        popt, _ = curve_fit(gaussian, x, y, p0=p0, bounds=bounds, maxfev=5000)
        _, mu, sigma = popt
        fwhm = 2.355 * abs(sigma)
        return float(mu), float(fwhm)
    except RuntimeError:
        return np.nan, np.nan

peak_records = []
for sample in ['BSA', 'BSA+TCEP']:
    names = sample_names_dict[sample]
    container = sample_containers_dict[sample]
    ax = container.spectral_axis
    for i, name in enumerate(names):
        if 'BSA' not in name:
            continue
        for ph in ph_to_plot:
            if not name.endswith(f"pH {ph}"):
                continue
            row = container.spectral_data[i]
            center_1657, fwhm_1657 = fit_gaussian_peak(ax, row, 1657)
            peak_records.append({
                'pH': ph,
                'sample': sample,
                'ratio_1320_1266': intensity_at(ax, row, 1320) / intensity_at(ax, row, 1266),
                # 'i1320': intensity_at(ax, row, 1320) / intensity_at(ax, row, 1002),
                'i1657': intensity_at(ax, row, 1657) / intensity_at(ax, row, 1002),
                'i1657_center': center_1657,
                'i1657_fwhm': fwhm_1657,
                'i1410': intensity_at(ax, row, 1410) / intensity_at(ax, row, 1002),
                'i932': intensity_at(ax, row, 932) / intensity_at(ax, row, 1002),
            })

peak_df = pd.DataFrame(peak_records)

peak_params = [
    ('i1657',          'Amide I (1657 cm⁻¹ / 1,002 cm⁻¹)'),
    ('i1657_center',   'Amide I center'),
    ('i1657_fwhm',     'Amide I FWHM'),
    # ('i1320',          'C⍺-H deformation (1320 cm⁻¹ / 1,002 cm⁻¹)'),
    ('ratio_1320_1266','Amide III (1320 cm⁻¹ / 1266 cm⁻¹)'),
    ('i932',           'C⍺-C stretch (932 cm⁻¹ / 1,002 cm⁻¹)'),
    ('i1410',          'Carboxylate (1410 cm⁻¹ / 1,002 cm⁻¹)'),
]

n_cols = 3
n_rows = int(np.ceil(len(peak_params) / n_cols))
sample_colors = {'BSA': BSA_color, 'BSA+TCEP': BSA_TCEP_color}

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[label for _, label in peak_params],
                    vertical_spacing=0.3,
                    horizontal_spacing=0.1)
for idx, (col, label) in enumerate(peak_params):
    row_idx, col_idx = divmod(idx, n_cols)
    for sample in ['BSA', 'BSA+TCEP']:
        df_s = peak_df[peak_df['sample'] == sample].sort_values('pH')
        fig.add_trace(
            go.Scatter(
                x=df_s['pH'], y=df_s[col],
                mode='lines+markers',
                name=sample,
                legendgroup=sample,
                showlegend=(idx == 0),
                line=dict(color=sample_colors[sample]),
            ),
            row=row_idx + 1, col=col_idx + 1,
        )
fig.update_layout(
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1)
)
fig.update_xaxes(title_text='pH')
apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")
fig.show()

fig.write_image(plot_dir_pdf / "peaks_vs_pH.pdf", scale=plot_scale)
fig.write_html(plot_dir_htmls / "peaks_vs_pH.html")